# 13 — Three-way comparison, 2023-04-05, and the co-located cross-sensor test

**Scientific question.** On the one date when all three systems observed, what do
they say — and what is the *like-for-like* SWOT ↔ ICESat-2 difference?

**Terminology.** The triple is **SAME-DAY, NOT SIMULTANEOUS**: Δt(SWOT − ICESat-2)
= 14.22 h and the gauge is a daily value with no epoch.

**Bug found by the figure work.** ICESat-2 has **zero** ATL13 segments within 10 km
of the Kherson gauge in the 2023 pre-breach window. Comparing gauge-collocated SWOT
against AOI-scale ICESat-2 mixes in ~30 km of channel gradient. The fix is to match
the two sensors *to each other* (500 m) and leave the gauge out.

In [1]:
# --- provenance: environment, git SHA and input hashes -----------------------
import hashlib, json, subprocess, sys, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd, geopandas as gpd, pyproj, rasterio, matplotlib
from swot_dnipro import config as CFG

np.random.seed(CFG.SEED)

def sha256(p, cap=64 * 1024 * 1024):
    p = Path(p)
    if not p.exists() or p.stat().st_size > cap:
        return f"(skipped: {'missing' if not p.exists() else 'too large'})"
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()[:16]

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       cwd=CFG.ROOT, text=True).strip()
    except Exception:
        return "(not a git checkout)"

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("geopandas   ", gpd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("pyproj/PROJ ", pyproj.__version__, "/", pyproj.proj_version_str)
print("rasterio/GDAL", rasterio.__version__, "/", rasterio.__gdal_version__)
print("git SHA     ", git_sha())
print("seed        ", CFG.SEED)
print()
for p in [CFG.EGG2015_TIF, CFG.UA2019Z_ASC, CFG.CORRECTOR_BY_STATION,
          CFG.KHERSON_GAUGE_PARQUET, CFG.KHERSON_ATL13]:
    print(f"  {sha256(p)}  {Path(p).name}")


python       3.12.3
numpy        2.5.2
pandas       3.0.5
geopandas    1.1.4
matplotlib   3.11.1
pyproj/PROJ  3.7.2 / 9.5.1
rasterio/GDAL 1.5.1 / 3.12.4
git SHA      7d9580b
seed         42

  (skipped: too large)  egg_2015.tif
  297440243e2ed8d6  ua_2019z.asc
  92d9fd71a1c8da39  egg2015_to_evrf2019_by_station.csv
  3380b29a0d68016b  80805_yearbook.parquet
  1deb7b5cfd362bd7  kherson_atl13_pass_levels.parquet


In [2]:
seg = pd.read_parquet(CFG.ICESAT_ROOT / "data/processed/kherson_atl13_segments.parquet")
seg["dt"] = pd.to_datetime(seg["time"], utc=True, errors="coerce").dt.tz_localize(None)
from swot_dnipro.vertical import haversine_km
seg["dist_km"] = haversine_km(seg.lon, seg.lat, CFG.KHERSON_GAUGE[2], CFG.KHERSON_GAUGE[3])
pre = seg[(seg.dt >= "2023-01-01") & (seg.dt < CFG.BREACH_DATE)]
for r in (0.5, 1, 2, 5, 10, 20):
    print(f"  ATL13 segments within {r:4.1f} km of the gauge, 2023 pre-breach: "
          f"{int((pre.dist_km < r).sum()):5d}")

  ATL13 segments within  0.5 km of the gauge, 2023 pre-breach:     0
  ATL13 segments within  1.0 km of the gauge, 2023 pre-breach:     0
  ATL13 segments within  2.0 km of the gauge, 2023 pre-breach:     0
  ATL13 segments within  5.0 km of the gauge, 2023 pre-breach:     0
  ATL13 segments within 10.0 km of the gauge, 2023 pre-breach:     0
  ATL13 segments within 20.0 km of the gauge, 2023 pre-breach:  3393


In [3]:
subprocess.run([sys.executable, "../scripts/make_colocated.py"], check=True)

co-located pairs: 718 segments, 6 date x beam, 1 date(s)
  median SWOT - ICESat = -0.1786 m  NMAD 0.0115  CI95 [-0.2072, -0.1455]  n=6 beams
  |dt| median 14.22 h; 36.9 km from the gauge (both sensors)
      date  rgt beam  n_segments  H_ICESat_m  H_SWOT_m    diff_m  within_beam_nmad_m
2023-04-05  225 gt1l          41    0.592834  0.348329 -0.232157            0.031145
2023-04-05  225 gt1r           7    0.480994  0.347290 -0.124274            0.021254
2023-04-05  225 gt2l         138    0.636952  0.478645 -0.182313            0.024887
2023-04-05  225 gt2r          38    0.618761  0.449711 -0.166814            0.025954
2023-04-05  225 gt3l         352    0.641307  0.476251 -0.175554            0.036271
2023-04-05  225 gt3r         142    0.673001  0.481658 -0.181629            0.027274


CompletedProcess(args=['/home/niko/projects/icesat2-atl13-kakhovka/.venv/bin/python3', '../scripts/make_colocated.py'], returncode=0)

In [4]:
summ = pd.read_csv(CFG.FIGDATA / "Fig16_colocated_summary.csv")
beam = pd.read_csv(CFG.FIGDATA / "Fig16_colocated_pairs_perbeam.csv")
display(summ.T); display(beam)

,0
statistic,"SWOT - ICESat-2, spatially co-located"
match_radius_km,0.5
min_swot_pixels,30
median_diff_m,-0.178591
nmad_m,0.01149
ci95_low_m,-0.207235
ci95_high_m,-0.145544
n_independent_beams,6
n_segments,718
n_dates,1


,date,rgt,beam,swot_utc,icesat_utc,dt_hours,n_segments,median_dist_to_gauge_km,H_ICESat_m,H_SWOT_m,diff_m,within_beam_nmad_m
0,2023-04-05,225,gt1l,2023-04-05 21:26:15.859364,2023-04-05 07:12:58.836504576,14.221362,41,33.355398,0.592834,0.348329,-0.232157,0.031145
1,2023-04-05,225,gt1r,2023-04-05 21:26:15.859364,2023-04-05 07:12:59.176804864,14.221277,7,33.377779,0.480994,0.347290,-0.124274,0.021254
2,2023-04-05,225,gt2l,2023-04-05 21:26:15.859364,2023-04-05 07:12:58.948404736,14.221314,138,36.813788,0.636952,0.478645,-0.182313,0.024887
3,2023-04-05,225,gt2r,2023-04-05 21:26:15.859364,2023-04-05 07:12:59.298904576,14.221219,38,36.891012,0.618761,0.449711,-0.166814,0.025954
4,2023-04-05,225,gt3l,2023-04-05 21:26:15.859364,2023-04-05 07:12:59.170104832,14.221267,352,40.264495,0.641307,0.476251,-0.175554,0.036271
5,2023-04-05,225,gt3r,2023-04-05 21:26:15.859364,2023-04-05 07:12:59.524004608,14.221164,142,40.402483,0.673001,0.481658,-0.181629,0.027274


**Result.** Spatially co-located, same-day, gauge-free:
`H_SWOT − H_ICESat-2 = −0.179 m`, NMAD 0.011 m over 6 beams,
95 % CI [−0.207, −0.146] — the CI **excludes zero**.

**But** all six beams come from a single RGT on a single date, so they are
pseudo-replicates of one overpass, and Δt is 14.22 h. This is one clean
measurement, not six independent ones.